In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q datasets

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
class ReviewPairDataset(Dataset):
    def __init__(self, user_texts, item_texts, user_ids, item_ids, ratings, tokenizer, max_length=512):
        self.user_texts = user_texts
        self.item_texts = item_texts
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.user_texts)

    def __getitem__(self, idx):
        user_enc = self.tokenizer(self.user_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        item_enc = self.tokenizer(self.item_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        rating = torch.tensor(float(self.ratings[idx]), dtype=torch.float)

        return {
            "input_ids_user": user_enc["input_ids"].squeeze(0),
            "attention_mask_user": user_enc["attention_mask"].squeeze(0),
            "input_ids_item": item_enc["input_ids"].squeeze(0),
            "attention_mask_item": item_enc["attention_mask"].squeeze(0),
            "labels": rating,
            "user_id": user_id,
            "item_id": item_id
        }

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions  # 그대로 사용!

    # 지표 계산
    mae = mean_absolute_error(labels, preds)
    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)

    epsilon = 1e-8
    mape = np.mean(np.abs((labels - preds) / (labels + epsilon))) * 100

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape
    }

In [ ]:
class Co_Attention_Layer(nn.Module):
    def __init__(self, emb_dim=768, nhead=8):
        super().__init__()
        self.attn_user_to_item = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)
        self.attn_item_to_user = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)

    def forward(self, user_vec, item_vec):
        # 입력: [batch, emb]
        user_vec = user_vec.unsqueeze(1)  # [batch, 1, emb]
        item_vec = item_vec.unsqueeze(1)  # [batch, 1, emb]

        # ① user attends to item (Q=user, K/V=item)
        attn_u2i, _ = self.attn_user_to_item(query=user_vec, key=item_vec, value=item_vec)  # [batch, 1, emb]

        # ② item attends to user (Q=item, K/V=user)
        attn_i2u, _ = self.attn_item_to_user(query=item_vec, key=user_vec, value=user_vec)  # [batch, 1, emb]

        # Flatten and concat both directions
        concat = torch.cat([attn_u2i.squeeze(1), attn_i2u.squeeze(1)], dim=-1)  # [batch, emb*2]
        return concat

# 데이터셋 로딩

In [5]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_reasoning_v5_final.pkl')
print(df.shape)
df.head()

(66718, 11)


# Prepare Input

In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

# fit-transform
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 실험 데이터의 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

In [ ]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [8]:
train.shape, valid.shape, test.shape

((46702, 11), (6672, 11), (13344, 11))

In [9]:
n_users, n_items

(7449, 16099)

In [ ]:
# train, test, valid 변수 선언
train_user_id = train['user_id'].to_numpy()
train_item_id = train['asin'].to_numpy()
train_summary = train['LLaMA_Output']
train_image_description = train['LLaVA_Output']
train_rating = train['rating'].to_numpy()

valid_user_id = valid['user_id'].to_numpy()
valid_item_id = valid['asin'].to_numpy()
valid_summary = valid['LLaMA_Output']
valid_image_description = valid['LLaVA_Output']
valid_rating = valid['rating'].to_numpy()

test_user_id = test['user_id'].to_numpy()
test_item_id = test['asin'].to_numpy()
test_summary = test['LLaMA_Output']
test_image_description = test['LLaVA_Output']
test_rating = test['rating'].to_numpy()

# Load Model

In [ ]:
# BERT 모델 Loading
# 해당 모델은 Layer 6개인 모델

# 사용자 텍스트 처리 모델
BERT_review = AutoModel.from_pretrained("bert-base-uncased")

# 아이템 텍스트 처리 모델
BERT_image = AutoModel.from_pretrained("bert-base-uncased")

# tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Two Tower BERT LoRA

```python
MAS_Rec(
  user_id: Embedding(num_users, 64)
  item_id: Embedding(num_items, 64)
  user_model: BertModel
  item_model: BertModel
  cls_dense: MLP(768→512→256)
  co_attn: Co_Attention_Layer(MultiheadAttention*2, emb_dim=320)
  fc_final: MLP(640→256→1)
)

```

In [ ]:
def make_mlp(sizes, activation=nn.ReLU):
    """
    중간 feature extractor용 MLP
    모든 Linear 뒤에 activation 함수가 붙음 (마지막 레이어 포함)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
def make_last_mlp(sizes, activation=nn.ReLU):
    """
    최종 예측 MLP
    마지막 Linear 뒤에는 activation이 없음 (즉, 마지막은 Linear-only)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
class MAS_Rec(nn.Module):
    def __init__(self, n_users, n_items, user_model, item_model, nhead=8):
        super().__init__()

        # Interaction IDs
        self.user_id = nn.Embedding(n_users, 64)
        self.item_id = nn.Embedding(n_items, 64)

        # Two Tower Bert
        self.user_model = user_model
        self.item_model = item_model

        # [CLS] Dense
        self.cls_dense = make_mlp([768, 512, 256])

        # Co-Attention (bi-directional cross attention)
        # 256([CLS] dense output emb) + 64(interaction emb)
        self.cross_attn = Co_Attention_Layer(emb_dim=320, nhead=nhead)

        # Last Dense
        self.fc_final = make_last_mlp([640, 256, 1])

    def forward(self, user_id, item_id, input_ids_user, attention_mask_user, input_ids_item, attention_mask_item, labels=None):

        # Interaction IDs
        user_emb = self.user_id(user_id)    # [batch, 64]
        item_emb = self.item_id(item_id)    # [batch, 64]

        # Two Tower BERT
        # 여기에서 input_ids는 token_id(token index)
        user_cls = self.user_model(input_ids=input_ids_user, attention_mask=attention_mask_user).last_hidden_state[:, 0, :]
        item_cls = self.item_model(input_ids=input_ids_item, attention_mask=attention_mask_item).last_hidden_state[:, 0, :]

        # [CLS] Dense
        user_cls = self.cls_dense(user_cls)
        item_cls = self.cls_dense(item_cls)

        # Concat
        user_feat = torch.cat([user_emb, user_cls], dim=-1)
        item_feat = torch.cat([item_emb, item_cls], dim=-1)

        # Co-Attetnion
        co_attn_output = self.cross_attn(user_feat, item_feat)  # [batch, 512]

        # Last Dense Layer
        logits = self.fc_final(co_attn_output).squeeze(1)

        if labels is not None:
            loss = nn.MSELoss()(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

# Fine Tuning

In [ ]:
# 모델 생성
model = MAS_Rec(Num_User, Num_Item, user_model=BERT_review, item_model=BERT_image, nhead=8)

In [ ]:
for name, module in model.named_children():
    print(name, module)

In [ ]:
# 실제 훈련용
train_dataset = ReviewPairDataset(train_summary, train_image_description, train_user_id, train_item_id, train_rating, tokenizer, max_length=512)
valid_dataset = ReviewPairDataset(valid_summary, valid_image_description, valid_user_id, valid_item_id, valid_rating, tokenizer, max_length=512)
test_dataset = ReviewPairDataset(test_summary, test_image_description, test_user_id, test_item_id, test_rating, tokenizer, max_length=512)

In [18]:
# 하이퍼파라미터
num_training_samples = 46702 # 46702 # 16443

# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps = 4,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
)

# ✅ 학습 시작
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,9.046300,0.421454,0.384904,0.421454,0.649195,12.209223
2,0.261700,0.237677,0.303267,0.237677,0.487521,8.745333
3,0.220500,0.233154,0.297078,0.233154,0.482860,8.689302
4,0.202400,0.235845,0.300892,0.235845,0.485638,8.632676
5,0.193800,0.230336,0.297368,0.230336,0.479933,8.498744


TrainOutput(global_step=915, training_loss=1.9849564057230298, metrics={'train_runtime': 2378.49, 'train_samples_per_second': 98.176, 'train_steps_per_second': 0.385, 'total_flos': 0.0, 'train_loss': 1.9849564057230298, 'epoch': 5.0})

# Train

In [19]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps = 4,
    num_train_epochs=100,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # ✅ 여기에 patience 설정
)

# ✅ 학습 시작
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,0.195100,0.234410,0.293682,0.234410,0.484159,8.544450
2,0.172800,0.218270,0.275472,0.218270,0.467193,7.890491
3,0.164700,0.235279,0.300717,0.235279,0.485056,8.487975
4,0.171200,0.255697,0.308511,0.255697,0.505664,8.944203
5,0.157000,0.218129,0.281029,0.218129,0.467043,8.163017
6,0.131800,0.247059,0.291738,0.247059,0.497050,8.437413
7,0.116800,0.249600,0.316139,0.249600,0.499600,9.017208
8,0.114000,0.269387,0.326479,0.269387,0.519025,9.427096
9,0.101600,0.282034,0.336465,0.282034,0.531068,9.661720
10,0.105800,0.266153,0.334522,0.266153,0.515900,9.407026


TrainOutput(global_step=1830, training_loss=0.14198138814806285, metrics={'train_runtime': 4757.2697, 'train_samples_per_second': 981.698, 'train_steps_per_second': 3.847, 'total_flos': 0.0, 'train_loss': 0.14198138814806285, 'epoch': 10.0})

In [20]:
# 평가
results = trainer.evaluate()

# 평가 결과 출력
print("Evaluation Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Evaluation Results:
eval_loss: 0.2181
eval_mae: 0.2810
eval_mse: 0.2181
eval_rmse: 0.4670
eval_mape: 8.1630
eval_runtime: 31.7449
eval_samples_per_second: 210.1760
eval_steps_per_second: 3.3080
epoch: 10.0000


In [ ]:
# pd.set_option('display.max_colwidth', None)

# error_sample = df.loc[df['LLaMA_Output'].str.contains('Input')==True, ['title_y', 'text', 'LLaMA_Output']].tail(10)



In [ ]:
import numpy as np

# 성능 데이터 입력
results = {
    "Simple": {
        "MAE": [0.2881, 0.2792, 0.2692, 0.2720, 0.2701],
        "MSE": [0.2128, 0.2196, 0.2220, 0.2140, 0.2135],
        "RMSE": [0.4613, 0.4686, 0.4712, 0.4626, 0.4621],
        "MAPE": [0.0817, 0.0827, 0.0811, 0.0795, 0.0782],
    }
}

# 평균과 표준편차 계산
summary = {}
for model, metrics in results.items():
    summary[model] = {}
    for metric, values in metrics.items():
        values_arr = np.array(values)
        summary[model][metric] = {
            "mean": round(values_arr.mean(), 6),
            "std": round(values_arr.std(ddof=1), 6)  # 표본 표준편차
        }

# 결과 출력
for model, metrics in summary.items():
    print(f"Model: {model}")
    for metric, stats in metrics.items():
        print(f"  {metric} -> Mean: {stats['mean']}, Std: {stats['std']}")


Model: Simple
  MAE -> Mean: 0.27572, Std: 0.007958
  MSE -> Mean: 0.21638, Std: 0.004145
  RMSE -> Mean: 0.46516, Std: 0.004448
  MAPE -> Mean: 0.08064, Std: 0.001791
